# 10 — E3B | Semantic Ablation: Popularity Bias Analysis

**Project:** SNA-MXH · Group 14 · Amazon · RQ2  
**Purpose:** Compare the accuracy–popularity trade-off across **RLMRec-Con Full**, **User-only**, and **Item-only** (3 seeds each).

### What this notebook does
1. Mounts Google Drive and locates the original training & test interaction matrices.
2. Uses the **same E2 popularity protocol**: train-only candidate masking; Head = top 20% by binary train-degree, ties broken by item ID; Tail = remainder.
3. Reads **9 previously trained E1/E3A Top-20 exports** (no model retraining).
4. Recomputes accuracy and popularity metrics for K=10 and K=20, checks E2 Full rows and saved trainer metrics.
5. Exports seed results, mean ± sample SD (ddof=1), comparison deltas and plots for RQ2.

**Naming note:** `08_E3A_...` is the ablation *training* notebook; this file is the next-stage E3B *analysis* notebook. The underlying models remain E3A runs. `09_Audit.ipynb` should be completed first.

**Data integrity:** Read-only inputs. Outputs are written **only** to `experiments/E3B/popularity_bias/`. This export/data audit does **not** prove the source model code never used validation/test labels for training or ranking; that requires separate source-code review.

In [ ]:
# =========================== CELL 1: SETUP ===========================
from pathlib import Path
import os, json, pickle, math, csv, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr, pearsonr
from IPython.display import display
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('[INFO] Not in Colab; using local paths.')

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
# If auto-detection fails, set these two strings to paths inside your own Drive:
TRAIN_PATH = None       # e.g. '/content/drive/MyDrive/.../trn_mat.pkl'
TEST_PATH = None        # e.g. '/content/drive/MyDrive/.../tst_mat.pkl'

OUT = ROOT/'experiments'/'E3B'/'popularity_bias'
N_USERS, N_ITEMS = 11000, 9332
N_TEST_USERS = 9860
KS = (10, 20)
SEEDS = (2023, 2024, 2025)
HEAD_FRAC = 0.20
TOL = 1e-5
E2_DIR = ROOT/'experiments'/'E2'/'popularity_bias'

assert ROOT.is_dir(), f'Project ROOT not found: {ROOT}. Check Google Drive mount.'
OUT.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Output only   :', OUT)
print('K             :', KS)
print('Seeds         :', SEEDS)


## 1. Locate train/test matrices and fix E2 Head–Tail split

`trn_mat.pkl` / `tst_mat.pkl` are the **original split matrices**. Use binary positive interactions (`nonzero > 0`). The Head set consists of `ceil(0.2 × 9332) = 1867` items, ranked by descending **train degree**, then ascending item ID. This must match the stored E2 item-popularity table exactly.

In [ ]:
# ================= CELL 2: DATA LOADING / E2 PROTOCOL =================
TRAIN_NAMES = {'trn_mat.pkl', 'train_mat.pkl', 'trn_mat.npz', 'train_mat.npz'}
TEST_NAMES  = {'tst_mat.pkl', 'test_mat.pkl', 'tst_mat.npz', 'test_mat.npz'}

def find_matrix(root: Path, names: set, explicit=None):
    if explicit:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(f'Configured path missing: {p}')
        return p
    found = []
    guesses = [root/'datasets'/'amazon', root/'data'/'amazon',
               root/'dataset'/'amazon', root/'RLMRec'/'datasets'/'amazon',
               root/'RLMRec'/'data'/'amazon', root/'RLMRec'/'dataset'/'amazon',
               root/'amazon', root]
    for folder in guesses:
        for fname in sorted(names):
            p = folder/fname
            if p.is_file() and p not in found:
                found.append(p)
    excluded = {'experiments', 'checkpoints', 'logs', '.git', '__pycache__',
                'node_modules', '.ipynb_checkpoints'}
    for path, dirs, files in os.walk(root):
        rel_depth = len(Path(path).relative_to(root).parts)
        dirs[:] = [] if rel_depth >= 8 else [d for d in dirs if d not in excluded]
        for fn in files:
            if fn in names:
                p = Path(path)/fn
                if p not in found:
                    found.append(p)
    if not found:
        raise FileNotFoundError('Could not find ' + ', '.join(sorted(names)) +
                                ' under ROOT. Set TRAIN_PATH or TEST_PATH in Cell 1.')
    found.sort(key=lambda p: ('amazon' not in str(p).lower(), len(str(p))))
    if len(found) > 1:
        print('[INFO] Matrix candidates; using the first match:')
        for p in found: print(' -', p)
    return found[0]

def load_binary_matrix(path: Path):
    if path.suffix == '.npz':
        mat = sparse.load_npz(path)
    else:
        # Read only your own trusted project files (pickle is executable).
        with path.open('rb') as f:
            mat = pickle.load(f)
    if sparse.issparse(mat):
        mat = mat.tocsr(copy=True)
    elif isinstance(mat, np.ndarray) and mat.ndim == 2:
        mat = sparse.csr_matrix(mat)
    else:
        raise TypeError(f'Unsupported matrix format in {path}: {type(mat)}')
    assert mat.shape == (N_USERS, N_ITEMS), (str(path), mat.shape)
    mat.sum_duplicates()
    mat.eliminate_zeros()
    mat.data = np.ones(mat.nnz, dtype=np.uint8)
    mat.sort_indices()
    return mat

trn_path = find_matrix(ROOT, TRAIN_NAMES, TRAIN_PATH)
tst_path = find_matrix(ROOT, TEST_NAMES, TEST_PATH)
trn = load_binary_matrix(trn_path)
tst = load_binary_matrix(tst_path)
print('[TRAIN]', trn_path, 'nonzero=', trn.nnz)
print('[TEST ]', tst_path, 'nonzero=', tst.nnz)

expected_users = np.flatnonzero(np.diff(tst.indptr) > 0)
assert len(expected_users) == N_TEST_USERS, (len(expected_users), N_TEST_USERS)

degree = np.asarray(trn.sum(axis=0)).ravel().astype(np.int64)
n_head = math.ceil(HEAD_FRAC*N_ITEMS)
sorted_items = np.lexsort((np.arange(N_ITEMS), -degree))
is_head = np.zeros(N_ITEMS, dtype=bool)
is_head[sorted_items[:n_head]] = True
is_tail = ~is_head
print(f'Head: {is_head.sum()} | Tail: {is_tail.sum()} | total: {N_ITEMS}')
assert (is_head.sum(), is_tail.sum()) == (1867, 7465)

# Independent cross-check against the E2 file, if available.
e2_pop_path = E2_DIR/'E2_item_popularity.csv'
if e2_pop_path.is_file():
    e2_pop = pd.read_csv(e2_pop_path).sort_values('item_id')
    assert np.array_equal(e2_pop['item_id'].to_numpy(), np.arange(N_ITEMS))
    assert np.array_equal(e2_pop['train_degree'].to_numpy(), degree), 'E2 train degrees differ!'
    assert np.array_equal(e2_pop['group'].to_numpy() == 'Head', is_head), 'E2 Head/Tail split differs!'
    print('[PASS] Head/Tail split matches E2_item_popularity.csv exactly.')
else:
    print('[WARN] E2_item_popularity.csv missing; using original train matrix and E2 rule.')

# Precompute group-specific positive counts for macro group recalls.
tst_pos_per_user = np.diff(tst.indptr).astype(np.int64)
row_of_positive = np.repeat(np.arange(N_USERS), tst_pos_per_user)
head_pos = np.bincount(row_of_positive,
                       weights=is_head[tst.indices].astype(np.int64),
                       minlength=N_USERS).astype(np.int64)
tail_pos = tst_pos_per_user - head_pos
n_head_eligible = int(np.count_nonzero(head_pos[expected_users]))
n_tail_eligible = int(np.count_nonzero(tail_pos[expected_users]))
print(f'Eligible test users — Head={n_head_eligible} | Tail={n_tail_eligible}')
if (n_head_eligible, n_tail_eligible) != (7059, 7835):
    raise AssertionError('Head/Tail-eligible user counts differ from E2 protocol. STOP and investigate.')


## 2. Evaluate 9 existing Top-K exports

All models use the **same 9860 test users**, the same train-only candidate masking, and the same Head/Tail membership from E2. For group-specific recall, average only across test users with at least one positive item in that group (conditional macro). Recomputed Recall/NDCG must match E1/E3A trainer metrics within `1e-5`.

In [ ]:
# ======================= CELL 3: EVALUATION =========================
def input_runs():
    pred = ROOT/'experiments'/'predictions'/'lightgcn_plus'
    e3 = ROOT/'experiments'/'E3'/'runs'
    for seed in SEEDS:
        yield ('RLMRec-Con Full', seed, pred/f'seed_{seed}'/'topk_test.npz',
               ROOT/'experiments'/'E1'/'lightgcn_plus'/f'seed_{seed}'/'metrics.json')
    for mode, label in [('user_only', 'User-only'), ('item_only', 'Item-only')]:
        for seed in SEEDS:
            folder = e3/mode/'weight_e1_default'/f'seed_{seed}'
            yield (label, seed, folder/'topk'/'topk_test.npz', folder/'metrics.json')

def load_topk(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing Top-K export: {path}')
    with np.load(path, allow_pickle=False) as z:
        users = z['user_ids']
        items = z['item_ids']
        scores = z['scores']
    assert users.shape == (N_TEST_USERS,)
    assert items.shape == (N_TEST_USERS, 20) and scores.shape == items.shape
    assert users.dtype.kind in 'iu' and items.dtype.kind in 'iu'
    assert np.array_equal(np.sort(users), expected_users), f'Different test users: {path}'
    assert np.all((items >= 0)&(items < N_ITEMS))
    assert np.all(np.diff(np.sort(items, axis=1), axis=1) != 0), 'Duplicate item in Top-K'
    assert np.all(np.isfinite(scores))
    assert np.all(np.diff(scores, axis=1) <= 1e-6), 'Scores are not descending'
    assert not np.asarray(trn[np.repeat(users, 20), items.ravel()]).ravel().any(), 'Train collision'
    return users, items

def compute_metrics(users, items, k):
    recs = items[:, :k]
    n = len(users)
    exp_degree = degree[recs]
    tail_rec = is_tail[recs]
    hits = np.asarray(tst[np.repeat(users, k), recs.ravel()]).reshape(n, k).astype(bool)
    user_truth = tst_pos_per_user[users]
    assert np.all(user_truth > 0)
    # Accuracy: macro Recall, NDCG with binary test labels (same as E1).
    recalls = hits.sum(axis=1)/user_truth
    discount = 1/np.log2(np.arange(2, k+2))
    idcg_prefix = np.r_[0., np.cumsum(discount)]
    dcg = hits @ discount
    idcg = idcg_prefix[np.minimum(user_truth, k)]
    ndcg = dcg/idcg
    hden, tden = head_pos[users], tail_pos[users]
    head_hits = (hits & is_head[recs]).sum(axis=1)
    tail_hits = (hits & is_tail[recs]).sum(axis=1)
    head_ok, tail_ok = hden > 0, tden > 0
    assert head_ok.sum() == n_head_eligible and tail_ok.sum() == n_tail_eligible
    # Degree/exposure correlation includes EVERY catalog item (even zero exposure).
    exposure = np.bincount(recs.ravel(), minlength=N_ITEMS)
    sp = spearmanr(degree, exposure)
    sp_corr = float(sp.statistic)
    sp_p = float(sp.pvalue)
    log_corr = float(pearsonr(np.log1p(degree), np.log1p(exposure)).statistic)
    return {
        'Recall': float(recalls.mean()), 'NDCG': float(ndcg.mean()),
        'ARP': float(exp_degree.mean()),
        'APLT': float(tail_rec.mean()),
        'HeadRecall': float((head_hits[head_ok]/hden[head_ok]).mean()),
        'TailRecall': float((tail_hits[tail_ok]/tden[tail_ok]).mean()),
        'HeadExposureShare': float(1-tail_rec.mean()),
        'Coverage': float((exposure > 0).sum()/N_ITEMS),
        'DegreeExposureSpearman': sp_corr,
        'DegreeExposureSpearmanP': sp_p,
        'DegreeExposureLogPearson': log_corr,
        'n_head_eligible_users': int(head_ok.sum()),
        'n_tail_eligible_users': int(tail_ok.sum()),
    }

def check_against_trainer(model, k, metrics, path):
    if not path.is_file():
        raise FileNotFoundError(f'Trainer metrics missing for {model}: {path}')
    with path.open(encoding='utf-8') as f:
        saved = json.load(f)
    trainer = saved.get('test_metrics', saved)
    for label, key in [('Recall', f'recall@{k}'), ('NDCG', f'ndcg@{k}')]:
        if key not in trainer:
            raise KeyError(f'Missing saved {key}: {path}')
        difference = abs(metrics[label] - float(trainer[key]))
        if difference > TOL:
            raise AssertionError(f'{model} k={k} {key}: export={metrics[label]:.8f} vs trainer={trainer[key]:.8f} diff={difference:.7g}')

rows=[]
ref_user_order = None
for model, seed, topk_path, trainer_path in input_runs():
    users, items = load_topk(topk_path)
    if ref_user_order is None:
        ref_user_order = users.copy()
    else:
        # Membership guaranteed; if order differs, all per-user stats still use user IDs.
        if not np.array_equal(users, ref_user_order):
            print(f'[INFO] User order differs in {model} {seed}; user IDs used safely.')
    for k in KS:
        m = compute_metrics(users, items, k)
        check_against_trainer(model, k, m, trainer_path)
        rows.append({'model': model, 'seed': seed, 'K': k,
                     'n_test_users': len(users), **m, 'topk_file': str(topk_path)})
    print(f'[PASS] {model:20s} seed={seed} | Recall@20={rows[-1]["Recall"]:.6f} | ARP@20={rows[-1]["ARP"]:.3f} | APLT@20={rows[-1]["APLT"]:.3%}')

seed_df = pd.DataFrame(rows)
assert len(seed_df)==18 and seed_df.groupby(['model','K']).size().eq(3).all()
print('\n[PASS] 9/9 Top-K runs and 18 (run, K) evaluations verified against trainer accuracy.')
display(seed_df[seed_df.K==20][['model','seed','Recall','NDCG','ARP','APLT','HeadRecall','TailRecall']])


## 3. Validate Full RLMRec-Con against E2 and create mean ± SD tables

The E2 Full model provides a consistency bridge. If its metrics do not match E2 at the same seed/K, this notebook stops rather than silently comparing incompatible definitions.

In [ ]:
# ================= CELL 4: REFERENCE CHECK + SUMMARY =================
e2_seed_path = E2_DIR/'E2_seed_metrics.csv'
assert e2_seed_path.is_file(), f'E2 reference file missing: {e2_seed_path}'
e2_seed = pd.read_csv(e2_seed_path)
metric_compare = ['ARP','APLT','HeadRecall','TailRecall','HeadExposureShare',
                  'Coverage','DegreeExposureSpearman','DegreeExposureLogPearson']
diff_records=[]
for _, row in seed_df[seed_df.model=='RLMRec-Con Full'].iterrows():
    match = e2_seed[(e2_seed.model=='RLMRec-Con') &
                    (e2_seed.seed.astype(str)==str(row.seed)) &
                    (e2_seed.K==row.K)]
    assert len(match)==1, f'E2 run missing: {row.seed} K={row.K}'
    ref = match.iloc[0]
    for metric in metric_compare:
        delta = abs(float(row[metric])-float(ref[metric]))
        diff_records.append({'seed':int(row.seed),'K':int(row.K),
                             'metric':metric,'abs_diff':delta})
        if not np.isfinite(delta) or delta>TOL:
            raise AssertionError(f'E2 PROTOCOL MISMATCH {metric} seed={row.seed} K={row.K}: new={row[metric]:.8f} E2={ref[metric]:.8f} delta={delta:.8g}')
print('[PASS] E3B Full recomputation matches E2 popularity protocol.')
print('Largest abs diff:', max(x['abs_diff'] for x in diff_records))

metrics=['Recall','NDCG','ARP','APLT','HeadRecall','TailRecall',
         'HeadExposureShare','Coverage','DegreeExposureSpearman','DegreeExposureLogPearson']
agg=seed_df.groupby(['model','K'],sort=False)[metrics].agg(['mean','std'])
agg.columns=[f'{metric}_{stat}' for metric,stat in agg.columns]
summary=agg.reset_index()
summary.insert(2,'n_seeds',3)
summary.insert(3,'n_test_users',N_TEST_USERS)
print('\n----- E3B SUMMARY @20 (mean ± sample SD, ddof=1) -----')
display(summary[summary.K==20][['model','Recall_mean','Recall_std','ARP_mean','ARP_std',
                                'APLT_mean','APLT_std','HeadRecall_mean','TailRecall_mean']])

# Comparison against Full for each K. Deltas are (variant - Full).
deltas=[]
for k in KS:
    t=summary[summary.K==k]
    full=t[t.model=='RLMRec-Con Full'].iloc[0]
    for _, r in t.iterrows():
        d={'model':r.model,'K':k}
        for m in metrics:
            d[f'{m}_mean']=float(r[f'{m}_mean'])
            d[f'Delta_{m}_vs_Full']=float(r[f'{m}_mean']-full[f'{m}_mean'])
        deltas.append(d)
tradeoff=pd.DataFrame(deltas)
display(tradeoff[tradeoff.K==20][['model','Recall_mean','ARP_mean','APLT_mean',
                                 'Delta_Recall_vs_Full','Delta_ARP_vs_Full',
                                 'Delta_APLT_vs_Full']])


## 4. RQ2 figures and export artifacts

**Interpretation discipline:** A lower ARP or higher APLT is **not sufficient** to claim an improvement; compare Recall/NDCG and Tail Recall at the same time. Do not interpret a difference across only three seeds as statistically significant without appropriate further analysis.

In [ ]:
# =================== CELL 5: FIGURES / SAVE =====================
def plot_tradeoff(x_metric, x_label, filename):
    fig,ax=plt.subplots(figsize=(8,5.2))
    for _,r in summary[summary.K==20].iterrows():
        label=r['model']
        ax.errorbar(r[f'{x_metric}_mean'],r['Recall_mean'],
                    xerr=r[f'{x_metric}_std'], yerr=r['Recall_std'],
                    fmt='o',capsize=4,label=label)
        ax.annotate(label, (r[f'{x_metric}_mean'],r['Recall_mean']),xytext=(6,5),textcoords='offset points',fontsize=9)
    ax.set_xlabel(x_label)
    ax.set_ylabel('Recall@20 (higher is better)')
    ax.set_title('E3B / RQ2 — Accuracy–Popularity Trade-off')
    ax.grid(alpha=.25)
    fig.tight_layout()
    fig.savefig(OUT/f'{filename}.png',dpi=180)
    fig.savefig(OUT/f'{filename}.pdf')
    plt.show()

plot_tradeoff('ARP','ARP@20 — average train degree (lower = less popular items)',
              'E3B_Recall20_vs_ARP20')
plot_tradeoff('APLT','APLT@20 — share of tail recommendations (higher = more tail)',
              'E3B_Recall20_vs_APLT20')

fig,ax=plt.subplots(figsize=(8,5.2))
for _,r in summary[summary.K==20].iterrows():
    ax.errorbar(r['HeadRecall_mean'], r['TailRecall_mean'],
                xerr=r['HeadRecall_std'], yerr=r['TailRecall_std'],
                fmt='o',capsize=4,label=r['model'])
    ax.annotate(r['model'], (r['HeadRecall_mean'],r['TailRecall_mean']),
                xytext=(5,5), textcoords='offset points',fontsize=9)
ax.set_xlabel('Conditional Head Recall@20')
ax.set_ylabel('Conditional Tail Recall@20')
ax.set_title('E3B / RQ2 — Head versus Tail Recall')
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(OUT/'E3B_Head_vs_Tail_Recall20.png',dpi=180)
fig.savefig(OUT/'E3B_Head_vs_Tail_Recall20.pdf')
plt.show()

protocol={
    'project':'SNA-MXH','stage':'E3B_Popularity_Bias_Analysis','research_question':'RQ2',
    'source_runs':'E1 RLMRec-Con Full (3 seeds) and E3A User-only/Item-only (3 seeds each)',
    'dataset':'amazon','train_matrix':str(trn_path),'test_matrix':str(tst_path),
    'train_mask':'train_only','validation_masked':False,'test_labels_used_for_ranking':False,
    'k_values':list(KS),'seeds':list(SEEDS),
    'n_users':N_USERS,'n_items':N_ITEMS,'n_test_users':N_TEST_USERS,
    'head_fraction':HEAD_FRAC,'head_count_rule':'ceil(0.2*n_items)',
    'tie_break':'ascending item ID','popularity_source':'binary train interactions only',
    'arp':'macro mean train degree of Top-K items',
    'aplt':'macro share of Tail items in Top-K',
    'head_tail_recall':'conditional macro, eligible users have >=1 group-specific test positive',
    'degree_exposure':'catalog-wide including zero-exposure items',
    'accuracy':'macro Recall@K and NDCG@K with test positives',
    'aggregate':'mean and sample SD (ddof=1)',
    'e2_protocol_match':True,'trainer_accuracy_match':True,
    'generated_at_utc':datetime.now(timezone.utc).isoformat()
}
seed_df.to_csv(OUT/'E3B_seed_metrics.csv',index=False)
summary.to_csv(OUT/'E3B_model_summary.csv',index=False)
tradeoff.to_csv(OUT/'E3B_accuracy_popularity_tradeoff.csv',index=False)
pd.DataFrame(diff_records).to_csv(OUT/'E3B_E2_reference_check.csv',index=False)
(OUT/'E3B_protocol.json').write_text(json.dumps(protocol,indent=2,ensure_ascii=False),encoding='utf-8')

text_report=['E3B POPULARITY BIAS — RQ2 (K=20)',
             'All 9 exports passed train-mask / duplicate / user-set checks.',
             'All E1/E3A accuracy values match saved trainer metrics.',
             'Full RLMRec-Con popularity metrics match E2.','']
for _,r in summary[summary.K==20].iterrows():
    text_report.append(f"{r['model']}: Recall@20={r['Recall_mean']:.6f} ± {r['Recall_std']:.6f}; "
                       f"ARP@20={r['ARP_mean']:.4f} ± {r['ARP_std']:.4f}; "
                       f"APLT@20={r['APLT_mean']:.4f} ± {r['APLT_std']:.4f}; "
                       f"TailRecall@20={r['TailRecall_mean']:.6f} ± {r['TailRecall_std']:.6f}")
text_report += ['', 'Interpretation: compare Recall/NDCG and Tail Recall with ARP/APLT together.',
                'No statistical significance or causal claims inferred from three seeds.']
(OUT/'E3B_RQ2_summary.txt').write_text('\n'.join(text_report)+'\n',encoding='utf-8')
print('\n'+'='*77)
print('SUCCESS — E3B POPULARITY ANALYSIS, 9/9 RUNS, K=10 AND K=20')
print('Saved to:',OUT)
for p in sorted(OUT.iterdir()):print(' •',p.name)
print('='*77)


### Files for Midterm / RQ2 write-up
- `E3B_seed_metrics.csv` — 9 runs × K=10/20, individual metrics.
- `E3B_model_summary.csv` — mean ± SD across 3 seeds.
- `E3B_accuracy_popularity_tradeoff.csv` — contrast against Full model.
- `E3B_Recall20_vs_ARP20.png`, `E3B_Recall20_vs_APLT20.png`, `E3B_Head_vs_Tail_Recall20.png` — figures (PDF versions also saved).
- `E3B_E2_reference_check.csv` — exact metric consistency check versus E2.
- `E3B_protocol.json` + `E3B_RQ2_summary.txt` — reproducibility and a compact result snapshot.

If any assertion fails, **do not force it to pass**. Save the error message and inspect the raw split/metadata. A failed E2 check can indicate a changed matrix or metric definition.